# A FAIR² Mental Health Survey Dataset from Kilifi County, Kenya: Demonstrating AI-Ready Data Standards for Data Infrastructure in Africa Exploration with `mlcroissant`
This notebook provides a step-by-step guide for exploring and processing the FAIR² Mental Health Survey dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL.

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install -U mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the dataset URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.vcs2-05nj/fair2.json'

# Load the dataset (this will fetch the Croissant schema and prepare the dataset)
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata

print(f"{metadata.name}: {metadata.description}")
print(f"\nVersion: {getattr(metadata, 'version', 'N/A')}")
print(f"Published: {getattr(metadata, 'datePublished', 'N/A')}")
print(f"Identifier: {getattr(metadata, 'identifier', 'N/A')}")

## 2. Data Overview
Review available record sets, their `@id`s, and fields present in each record set.

In [ ]:
# List all available record sets with their @id and field ids
print("Available Record Sets:")
record_sets = dataset.record_sets
for rs in record_sets:
    print(f"- {rs['@id']} \n  Name: {rs.get('name', 'N/A')}")
    # List fields for each record set
    fields = rs.get('field', [])
    if isinstance(fields, dict):  # fix if single field
        fields = [fields]
    if fields:
        print("  Fields:")
        for field in fields:
            print(f"    - {field['@id']} (name: {field.get('name', 'N/A')})")
    print("")
# Choose the primary record set for further exploration
if len(record_sets) > 0:
    main_record_set_id = record_sets[0]['@id']
    print(f"Selected main record set: {main_record_set_id}")
else:
    raise ValueError('No record sets found in the dataset!')

## 3. Data Extraction
Load data from a specific record set into a DataFrame for analysis. All identifiers used are the `@id`s from the Croissant schema.

In [ ]:
# Extract data from all available record sets
dataframes = {}
record_set_ids = [rs['@id'] for rs in record_sets]
for rs_id in record_set_ids:
    records = list(dataset.records(record_set=rs_id))
    if records:
        df = pd.DataFrame(records)
        dataframes[rs_id] = df
        print(f"Loaded DataFrame for record set {rs_id} (rows: {df.shape[0]}, columns: {df.shape[1]})")
    else:
        print(f"No records found for record set {rs_id}.")

# Use the main record set for further analysis
primary_df = dataframes[main_record_set_id]
print(f"\nColumns in main record set ({main_record_set_id}):")
print(primary_df.columns.tolist())
primary_df.head()

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps, such as filtering records based on specific criteria, normalizing numeric fields, and grouping data. All column and field references use the associated Croissant `@id` identifiers.

In [ ]:
# Identify candidate numeric fields (by @id) from data overview
# We'll attempt to use the GAD-7 total score if it is present, for illustration.
candidate_numeric_fields = [col for col in primary_df.columns if 'gad_7' in col.lower() or 'phq_9' in col.lower() or 'score' in col.lower()]
if candidate_numeric_fields:
    numeric_field_id = candidate_numeric_fields[0]
else:
    # If such a field cannot be identified, select the first float/int field
    numeric_field_id = primary_df.select_dtypes(include=['number']).columns[0]

print(f"Using field: {numeric_field_id} as the numeric field for EDA.")

# Example threshold (could be adapted)
threshold = primary_df[numeric_field_id].quantile(0.75)  # Top quartile as an example
filtered_df = primary_df[primary_df[numeric_field_id] > threshold].copy()
print(f"Filtered records with {numeric_field_id} above the 75th percentile ({threshold:.2f}):")
print(filtered_df.head())

# Normalize numeric field
filtered_df[f"{numeric_field_id}_normalized"] = (filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()) / filtered_df[numeric_field_id].std()
print(f"\nNormalized {numeric_field_id}:")
print(filtered_df[[numeric_field_id, f"{numeric_field_id}_normalized"]].head())

# Choose a group field (e.g., gender, education), picking by Croissant @id
group_fields = [col for col in primary_df.columns if 'gender' in col.lower() or 'education' in col.lower() or 'village' in col.lower()]
if group_fields:
    group_field_id = group_fields[0]
    grouped_df = filtered_df.groupby(group_field_id)[numeric_field_id].mean().reset_index().rename(columns={numeric_field_id: f"mean_{numeric_field_id}"})
    print(f"\nAverage {numeric_field_id} grouped by {group_field_id}:")
    print(grouped_df.head())
else:
    print("No suitable group field found for grouping.")

## 5. Visualization
Visualize the distribution of the selected numeric field and mean score grouped by a demographic variable (if available).

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
sns.set(style='whitegrid')

# Histogram of the selected numeric field
plt.figure(figsize=(7,4))
sns.histplot(primary_df[numeric_field_id].dropna(), kde=True, bins=15)
plt.title(f'Distribution of {numeric_field_id}')
plt.xlabel(numeric_field_id)
plt.ylabel('Count')
plt.show()

# If grouping performed, show a barplot
if 'grouped_df' in locals() and group_fields:
    plt.figure(figsize=(7,4))
    sns.barplot(data=grouped_df, x=group_field_id, y=f"mean_{numeric_field_id}")
    plt.title(f'Mean {numeric_field_id} by {group_field_id}')
    plt.ylabel(f"Mean {numeric_field_id}")
    plt.xlabel(group_field_id)
    plt.show()

## 6. Conclusion
In this notebook, we loaded the FAIR² Mental Health Survey dataset via the Croissant schema using `mlcroissant`, inspected its record sets and fields by their `@id`, and performed exploratory data analysis using only `@id` references to remain schema-compliant. 

- We visualized the distribution of a key mental health score (such as GAD-7 or PHQ-9 total).
- We demonstrated filtering and normalization of numeric data, and grouping by demographics if available.

To extend your analysis, you can explore other record sets, fields, or perform more advanced transformations using the rich metadata provided by Croissant!